In [1]:
import pandas as pd

# NHGIS Data

## Data Setup

In [2]:
file_path = "data/raw/nhgis/nhgis0002_ts_nominal_county.csv"
df_nhgis = pd.read_csv(file_path, low_memory=False)

In [3]:
# Standardize variable format
df_nhgis.columns = df_nhgis.columns.str.lower()

In [4]:
# Drop unhelpful columns
col_to_drop = [
    'gisjoin',
    'state',
    'statenh',
    'county',
    'countynh', 
    'name',   # id variables

    'av0aam',
    'av1aam',
    'av1abm',
    'ar9aam',
    'b18aam',
    'b18abm',
    'b18acm',
    'b18adm',
    'b18aem',
    'a35aam',
    'b16aam',
    'b16abm',
    'ar5aam',
    'a68aam',
    'at5aam',
    'at5abm',
    'b69aam',
    'b69abm',
    'b69acm',
    'b84aam',
    'b84abm',
    'b84acm',
    'b84adm',
    'b84aem',
    'b84afm',
    'c98aam',
    'bd5aam',
    'cl6aam'
]

df_nhgis = df_nhgis.drop(columns=col_to_drop)

In [5]:
# Rename demographic and socio-economic variables for clarity
labels = {
    'statefp':'state_code',
    'countyfp':'county_code',
    
    'av0aa':'total_pop',
    'd15aa':'urban_pop',
    'd15ab':'rural_pop',
    'av1aa':'male_pop',
    'av1ab':'female_pop',
    'ar9aa':'median_age',
    'b18aa':'white_pop',
    'b18ab':'black_pop',
    'b18ac':'indig_pop',
    'b18ad':'asian_pop',
    'b18ae':'two_race_pop',
    'a35aa':'hispanic_pop',
    'b16aa':'family_pop',
    'b16ab':'non_family_pop',
    'ar5aa':'total_households',
    'a68aa':'total_families',
    'at5aa':'native_pop',
    'at5ab':'foreign_pop',
    'b69aa':'no_hs_pop',
    'b69ab':'hs_college_pop',
    'b69ac':'college_grad_pop',
    'b84aa':'labor_force_pop',
    'b84ab':'armed_force_pop',
    'b84ac':'civ_pop',
    'b84ad':'civ_employed_pop',
    'b84ae':'civ_unemployed_pop',
    'b84af':'out_labor_force_pop',
    'c98aa':'agg_travel_time_to_work',
    'bd5aa':'per_capita_income',   # noted as previous year
    'cl6aa':'pov_pop'
}

df_nhgis = df_nhgis.rename(columns=labels)

## Main Sample

## Interpolation Sample

5-year and 10-year data only and interpolation for individual years.

In [6]:
# Filter out socioeconomic variables not suitable for interpolation
socioeconomic_cols = [
    "civ_unemployed_pop",
    "civ_employed_pop",
    "pov_pop",
    "per_capita_income",
    "agg_travel_time_to_work"
]

df_nhgis_dem = df_nhgis.drop(
    columns=socioeconomic_cols,
    errors="ignore"
)

# Distinguish between id and value columns for later processing
id_cols = ['year', 'state_code', 'county_code']
value_cols = [c for c in df_nhgis_dem.columns if c not in id_cols]

In [7]:
# Pre-2000 data (decennial only)

df_pre_2000 = df_nhgis_dem.copy()
df_pre_2000['year'] = pd.to_numeric(df_pre_2000['year'], errors='coerce')
df_pre_2000 = df_pre_2000.loc[
    df_pre_2000['year'].between(1970, 2000) & (df_pre_2000['year'] % 10 == 0)
].copy()

# Interpolation function
def interp(g):
    state_code, county_code = g.name
    g = g.set_index('year').reindex(range(1970, 2001))
    vals = g[value_cols].apply(pd.to_numeric, errors='coerce').interpolate(method='linear')
    vals = vals.reset_index().rename(columns={'index': 'year'})
    vals['state_code'] = state_code
    vals['county_code'] = county_code
    return vals

df_pre_2000 = (
    df_pre_2000
    .groupby(['state_code', 'county_code'], group_keys=False)
    .apply(interp, include_groups=False)
)

In [8]:
# 2000-2008 data (decennial + 5-year of 2006-2010)
id_cols = ['year', 'state_code', 'county_code']

df_00_08 = df_nhgis_dem.copy()
value_cols = [c for c in df_00_08.columns if c not in id_cols]

year_str = df_00_08['year'].astype(str)
df_00_08['year'] = pd.to_numeric(year_str, errors='coerce')
df_00_08.loc[year_str.str.contains('2006-2010', na=False), 'year'] = 2008
df_00_08 = df_00_08.loc[df_00_08['year'].isin([2000, 2008])].copy()

def _interp_00_08(g):
    state_code, county_code = g.name
    g = g.set_index('year').reindex(range(2000, 2009))
    vals = g[value_cols].apply(pd.to_numeric, errors='coerce').interpolate(method='linear')
    vals = vals.reset_index().rename(columns={'index': 'year'})
    vals['state_code'] = state_code
    vals['county_code'] = county_code
    return vals

df_00_08 = (
    df_00_08
    .groupby(['state_code', 'county_code'], group_keys=False)
    .apply(_interp_00_08, include_groups=False)
)

In [9]:
# 2008-2023 data (5-year rolling average)
id_cols = ['year', 'state_code', 'county_code']
value_cols = [c for c in df_nhgis_dem.columns if c not in id_cols]

df_08_23 = df_nhgis_dem.copy()
year_str = df_08_23['year'].astype(str)
df_08_23['year'] = pd.to_numeric(year_str, errors='coerce')

is_range = year_str.str.contains(r'\d{4}[-–]\d{4}', na=False)
ranges = year_str.where(is_range).str.extract(r'(?P<start>\d{4})[-–](?P<end>\d{4})')
mid = (ranges['start'].astype(float) + ranges['end'].astype(float)) / 2

# Range-derived midpoints
df_ranges = df_08_23[is_range].copy()
df_ranges['year'] = mid
df_ranges['year'] = df_ranges['year'].round().astype('Int64')
df_ranges = df_ranges.loc[df_ranges['year'].between(2008, 2021)].copy()
df_ranges = df_ranges.loc[~df_ranges['year'].isin([2010, 2020])]

# Original single-year rows for 2010 and 2020
df_decennial = df_08_23[~is_range].copy()
df_decennial = df_decennial.loc[df_decennial['year'].isin([2010, 2020])]

df_08_23 = pd.concat([df_ranges, df_decennial], ignore_index=True)

In [10]:
# Combine year sections
df_nhgis_dem_all = (
    pd.concat([df_pre_2000, df_00_08, df_08_23], ignore_index=True)
    .sort_values(['state_code', 'county_code', 'year'])
    .reset_index(drop=True)
)

df_nhgis_dem_all['year'] = df_nhgis_dem_all['year'].astype('Int64').astype(str)

In [11]:
df_nhgis_dem_all.columns

Index(['year', 'total_pop', 'urban_pop', 'rural_pop', 'male_pop', 'female_pop',
       'median_age', 'white_pop', 'black_pop', 'indig_pop', 'asian_pop',
       'two_race_pop', 'hispanic_pop', 'family_pop', 'non_family_pop',
       'total_households', 'total_families', 'native_pop', 'foreign_pop',
       'no_hs_pop', 'hs_college_pop', 'college_grad_pop', 'labor_force_pop',
       'armed_force_pop', 'civ_pop', 'out_labor_force_pop', 'state_code',
       'county_code'],
      dtype='object')

In [12]:
df_nhgis_dem_all

,year,total_pop,urban_pop,rural_pop,male_pop,female_pop,median_age,white_pop,black_pop,indig_pop,...,foreign_pop,no_hs_pop,hs_college_pop,college_grad_pop,labor_force_pop,armed_force_pop,civ_pop,out_labor_force_pop,state_code,county_code
0,1970,24460.0,13116.0,11344.0,11947.0,12513.0,NaN,17511.0,6911.0,4.0,...,73.0,4298.0,7003.0,767.0,8692.0,352.0,8340.0,6655.0,1,1
1,1971,25239.9,13727.3,11512.6,12337.1,12902.8,NaN,18244.5,6942.3,13.3,...,93.2,4276.1,7427.1,902.0,9208.5,352.7,8855.8,6893.7,1,1
2,1972,26019.8,14338.6,11681.2,12727.2,13292.6,NaN,18978.0,6973.6,22.6,...,113.4,4254.2,7851.2,1037.0,9725.0,353.4,9371.6,7132.4,1,1
3,1973,26799.7,14949.9,11849.8,13117.3,13682.4,NaN,19711.5,7004.9,31.9,...,133.6,4232.3,8275.3,1172.0,10241.5,354.1,9887.4,7371.1,1,1
4,1974,27579.6,15561.2,12018.4,13507.4,14072.2,NaN,20445.0,7036.2,41.2,...,153.8,4210.4,8699.4,1307.0,10758.0,354.8,10403.2,7609.8,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
172100,2017,35428.0,NaN,NaN,16959.0,18469.0,43.9,26607.0,1346.0,159.0,...,NaN,4855.0,14424.0,6304.0,10695.0,0.0,10695.0,18925.0,72,153
172101,2018,34501.0,NaN,NaN,16484.0,18017.0,45.1,24931.0,1244.0,85.0,...,NaN,4194.0,14913.0,6156.0,10741.0,0.0,10741.0,18346.0,72,153
172102,2019,34704.0,NaN,NaN,16548.0,18156.0,45.5,24533.0,1282.0,48.0,...,NaN,3751.0,15259.0,6651.0,11471.0,0.0,11471.0,18015.0,72,153
172103,2020,34172.0,28084.0,6088.0,16251.0,17921.0,48.1,7312.0,1617.0,209.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,72,153


## Map to District Level

In [13]:
# Map county data to congressional districts

crosswalk = pd.read_csv("data/intermediate/county_to_cd_crosswalk.csv", dtype={
    'state_county_code': 'string',
    'district_code': 'string',
    'cd': 'string',
})
crosswalk['state_county_code'] = crosswalk['state_county_code'].str.zfill(5)
crosswalk['district_code'] = crosswalk['district_code'].str.zfill(2)
crosswalk['cd'] = crosswalk['cd'].str.zfill(3)

df_cd = df_nhgis_dem_all.copy()
df_cd['state_code'] = df_cd['state_code'].astype(str).str.zfill(2)
df_cd['county_code'] = df_cd['county_code'].astype(str).str.zfill(3)
df_cd['state_county_code'] = df_cd['state_code'] + df_cd['county_code']

df_cd = df_cd.merge(crosswalk, on='state_county_code', how='left', validate='many_to_many')

id_cols = ['year', 'state_code', 'district_code', 'cd']
value_cols = [c for c in df_cd.columns if c not in id_cols + ['county_code', 'state_county_code', 'afact']]

df_cd[value_cols] = df_cd[value_cols].apply(pd.to_numeric, errors='coerce')
df_cd[value_cols] = df_cd[value_cols].mul(df_cd['afact'], axis=0)

df_nhgis_dem_cd = (
    df_cd
    .groupby(id_cols, as_index=False)[value_cols]
    .sum()
)

In [14]:
df_nhgis_dem_cd

,year,state_code,district_code,cd,total_pop,urban_pop,rural_pop,male_pop,female_pop,median_age,...,total_families,native_pop,foreign_pop,no_hs_pop,hs_college_pop,college_grad_pop,labor_force_pop,armed_force_pop,civ_pop,out_labor_force_pop
0,1970,01,01,102,4.917470e+05,305907.0000,185840.0000,237827.000,2.539200e+05,0.0,...,118967.0000,489262.0000,2485.0000,91410.0000,1.417000e+05,17089.0000,172326.000,1326.0000,1.710000e+05,1.508250e+05
1,1970,01,01,103,4.723868e+05,304794.3456,167592.4288,228486.088,2.439007e+05,0.0,...,114810.7728,469907.1512,2479.6232,86881.9696,1.376891e+05,16531.5728,166882.440,1315.0256,1.655674e+05,1.443365e+05
2,1970,01,01,106,4.727529e+05,304927.5918,167825.3014,228663.914,2.440890e+05,0.0,...,114899.6584,470272.6261,2480.2671,86957.5388,1.377881e+05,16541.6834,167000.945,1315.3818,1.656856e+05,1.444528e+05
3,1970,01,01,108,4.623198e+05,301130.5614,161189.2822,223596.522,2.387233e+05,0.0,...,112366.7432,459857.9253,2461.9183,84804.0924,1.349663e+05,16253.5682,163623.985,1305.2314,1.623188e+05,1.411410e+05
4,1970,01,01,109,4.623198e+05,301130.5614,161189.2822,223596.522,2.387233e+05,0.0,...,112366.7432,459857.9253,2461.9183,84804.0924,1.349663e+05,16253.5682,163623.985,1305.2314,1.623188e+05,1.411410e+05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
293978,2021,56,00,118,5.797610e+05,0.0000,0.0000,296646.000,2.831150e+05,954.2,...,148988.0000,559397.0000,20364.0000,7492.0000,2.696760e+05,118336.0000,300480.000,3427.0000,2.970530e+05,1.617560e+05
293979,2021,56,00,119,5.797610e+05,0.0000,0.0000,296646.000,2.831150e+05,954.2,...,148988.0000,559397.0000,20364.0000,7492.0000,2.696760e+05,118336.0000,300480.000,3427.0000,2.970530e+05,1.617560e+05
293980,2021,72,98,117,3.254885e+06,0.0000,0.0000,1540987.000,1.713898e+06,3453.8,...,796974.0000,0.0000,0.0000,301849.0000,1.407562e+06,702653.0000,1269847.000,3360.0000,1.266487e+06,1.518078e+06
293981,2021,72,98,118,3.254885e+06,0.0000,0.0000,1540987.000,1.713898e+06,3453.8,...,796974.0000,0.0000,0.0000,301849.0000,1.407562e+06,702653.0000,1269847.000,3360.0000,1.266487e+06,1.518078e+06
